# Treccani DBI — floruit location & polity validation (sample of 10)

Builds the labeling dataset for the LLM-based verification method:

1. Sample 10 individuals with a Dizionario Biografico degli Italiani (Treccani) biography, stratified by **region of origin** × **floruit period**.
2. **Gemini 2.5 Flash, step 1** — extract the most granular floruit location, the floruit period, and verbatim evidence (original + English).
3. **Gemini 2.5 Flash, step 2** — map the location to the Cliopatria polity active during the floruit period.
4. Output a TSV with empty annotation columns (single annotator).

In [1]:
import json, os, random, time, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import duckdb
import pandas as pd
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv
from tqdm.auto import tqdm

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data" / "humans_clean.duckdb").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DB_PATH = PROJECT_ROOT / "data" / "humans_clean.duckdb"
OUT_DIR = PROJECT_ROOT / "annotations" / "treccani_validation"
CACHE_DIR = OUT_DIR / "_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
N_PER_BIN = 2                      # 2 individuals x 5 period bins = 10
MODEL = "google/gemini-2.5-flash"
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
BIO_CHAR_CAP = 25_000
PERIOD_BINS = [-10_000, 1300, 1500, 1650, 1800, 3000]
PERIOD_LABELS = ["<1300", "1300-1500", "1500-1650", "1650-1800", "1800+"]

load_dotenv(PROJECT_ROOT / ".env")
API_KEY = os.environ["OPEN_ROUTER_API"]
random.seed(SEED)

con = duckdb.connect(str(DB_PATH), read_only=True)
print("db:", DB_PATH.name, "| model:", MODEL)


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/homebrew/Cellar/python@3.10/3.10.13_1/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/homebrew/Cellar/python@3.10/3.10.13_1/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/homebrew/lib/python3.10/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/homebrew/lib/python3.

AttributeError: _ARRAY_API not found


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/opt/homebrew/Cellar/python@3.10/3.10.13_1/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/opt/homebrew/Cellar/python@3.10/3.10.13_1/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "/opt/homebrew/lib/python3.10/site-packages/ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "/opt/homebrew/lib/python3.

AttributeError: _ARRAY_API not found

db: humans_clean.duckdb | model: google/gemini-2.5-flash


## 1. Candidates — DBI individuals with floruit, Italian birthplace and a Cultura polity

In [2]:
candidates = con.execute("""
    WITH dbi AS (
        SELECT wikidata_id, any_value(value) AS dbi_id
        FROM identifiers WHERE property_id = 'P1986' AND value IS NOT NULL
        GROUP BY wikidata_id
    ),
    cultura AS (
        SELECT wikidata_id, string_agg(DISTINCT polity_name, '; ') AS cultura_polities
        FROM individuals_cliopatria GROUP BY wikidata_id
    )
    SELECT i.wikidata_id, i.name_en, d.dbi_id,
           fp.floruit_year, fp.floruit_period_start, fp.floruit_period_end,
           p.name_en AS birthplace, p.lat AS birth_lat,
           c.cultura_polities
    FROM individuals i
    JOIN dbi d       ON d.wikidata_id = i.wikidata_id
    JOIN individuals_floruit_period fp ON fp.wikidata_id = i.wikidata_id
    JOIN individuals_keys k ON k.wikidata_id = i.wikidata_id
    JOIN places p    ON p.id = k.birthcity_id
                    AND p.iso_a3_code = 'ITA' AND p.lat IS NOT NULL
    JOIN cultura c   ON c.wikidata_id = i.wikidata_id
    WHERE fp.floruit_period_start IS NOT NULL
      AND fp.floruit_period_end IS NOT NULL
      AND i.non_human = 0
      AND i.name_en IS NOT NULL
""").df()

print(f"candidates: {len(candidates):,}")
candidates.head()

candidates: 22,519


,wikidata_id,name_en,dbi_id,floruit_year,floruit_period_start,floruit_period_end,birthplace,birth_lat,cultura_polities
0,Q3769933,Giuseppe Alessandro Favaro,giuseppe-alessandro-favaro,<NA>,1909,1938,Revine,46.000278,Kingdom of Italy; Austria-Hungary
1,Q95147409,Giuseppe Graziani,giuseppe-graziani,<NA>,1728,1754,Centrale,45.727370,Republic of Venice
2,Q621087,Filippo Filippi,filippo-filippi,<NA>,1860,1887,Vicenza,45.550000,Kingdom of Sardinia; Kingdom of Italy
3,Q1528270,Giuseppe Damiani Almeyda,giuseppe-damiani-almejda,<NA>,1864,1896,Capua,41.105556,Kingdom of Italy
4,Q3772798,Goffredo Coppola,goffredo-coppola,<NA>,1932,1945,Guardia Sanframondi,41.250000,Kingdom of Italy; Nazi Germany


## 2. Stratified sample — region of origin × floruit period

In [3]:
# Region of origin from birthplace latitude (Italian macro-areas).
def region_of(lat):
    if lat >= 44.0:
        return "North"
    if lat >= 41.5:
        return "Center"
    return "South & Islands"

candidates["region"] = candidates["birth_lat"].apply(region_of)
candidates["period_bin"] = pd.cut(
    candidates["floruit_year"], bins=PERIOD_BINS, labels=PERIOD_LABELS)

print(candidates.groupby(["period_bin", "region"], observed=True).size().unstack(fill_value=0))

# 2 individuals per period bin, from different regions when possible.
rows = []
for label in PERIOD_LABELS:
    pool = candidates[candidates["period_bin"] == label]
    pool = pool.sample(frac=1, random_state=SEED)
    picked = pool.drop_duplicates("region").head(N_PER_BIN)
    if len(picked) < N_PER_BIN:                      # bin with a single region
        rest = pool.drop(picked.index).head(N_PER_BIN - len(picked))
        picked = pd.concat([picked, rest])
    rows.append(picked)

sample = pd.concat(rows).reset_index(drop=True)
assert len(sample) == 10 and sample["wikidata_id"].is_unique
sample["dbi_url"] = ("https://www.treccani.it/enciclopedia/"
                     + sample["dbi_id"] + "_(Dizionario-Biografico)/")
sample[["wikidata_id", "name_en", "region", "period_bin",
        "floruit_period_start", "floruit_period_end", "cultura_polities"]]

region      Center  North  South & Islands
period_bin                                
<1300           27     17               14
1300-1500       35     28                5
1500-1650       20     35                6
1650-1800       14     13                7
1800+            9     23                3


,wikidata_id,name_en,region,period_bin,floruit_period_start,floruit_period_end,cultura_polities
0,Q956946,John of Capua,South & Islands,<1300,1280,1310,Papal States
1,Q160424,Paul the Deacon,North,<1300,700,799,Alliance between Byzantine Empire and Khazaria...
2,Q2436996,Tito Livio Frulovisi,North,1300-1500,1430,1456,Papal States
3,Q3620361,Antonio da Pisa,Center,1300-1500,1384,1400,Republic of Pisa
4,Q3766550,Giovanni Battista Albanese,North,1500-1650,1580,1630,Republic of Venice
5,Q55226791,Litti Corbizzi,Center,1500-1650,1494,1515,Holy Roman Empire Minor States; Holy Roman Empire
6,Q55226863,Isabella Maria dal Pozzo,North,1650-1800,1674,1700,Duchy of Bavaria
7,Q3770142,Giuseppe Bonecchi,Center,1650-1800,1745,1777,Habsburg Monarchy
8,Q275985,Natalia Ginzburg,South & Islands,1800+,1946,1978,Republic of Italy
9,Q1062517,Giulio Aristide Sartorio,Center,1800+,1890,1922,Kingdom of Italy


## 3. Fetch the DBI biographies

The article body is embedded in the page's `__NEXT_DATA__` JSON payload.

In [4]:
PAGES_PATH = CACHE_DIR / "dbi_pages.jsonl"
UA = {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36"}


def fetch_dbi(row):
    r = requests.get(row["dbi_url"], headers=UA, timeout=30)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    nd = soup.find("script", id="__NEXT_DATA__")
    content = json.loads(nd.string)["props"]["pageProps"]["data"]["content"]
    text = BeautifulSoup(content, "html.parser").get_text(" ", strip=True)
    return {"wikidata_id": row["wikidata_id"], "dbi_url": row["dbi_url"], "text": text}


pages = {}
if PAGES_PATH.exists():
    for line in PAGES_PATH.open():
        r = json.loads(line)
        pages[r["wikidata_id"]] = r

todo = [row for _, row in sample.iterrows() if row["wikidata_id"] not in pages]
if todo:
    with ThreadPoolExecutor(max_workers=5) as ex, PAGES_PATH.open("a") as f:
        futs = [ex.submit(fetch_dbi, row) for row in todo]
        for fut in tqdm(as_completed(futs), total=len(futs), desc="fetch DBI"):
            r = fut.result()
            pages[r["wikidata_id"]] = r
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

assert all(qid in pages and len(pages[qid]["text"]) > 200 for qid in sample["wikidata_id"])
pd.DataFrame([{"wikidata_id": q, "chars": len(pages[q]["text"])} for q in sample["wikidata_id"]])

fetch DBI:   0%|          | 0/10 [00:00<?, ?it/s]

,wikidata_id,chars
0,Q956946,10889
1,Q160424,57496
2,Q2436996,26564
3,Q3620361,5526
4,Q3766550,2622
5,Q55226791,3532
6,Q55226863,4395
7,Q3770142,8279
8,Q275985,23142
9,Q1062517,19538


## 4. LLM step 1 — floruit location, floruit period, verbatim evidence

In [5]:
_tls = threading.local()


def _session():
    if not hasattr(_tls, "s"):
        s = requests.Session()
        s.headers.update({"Content-Type": "application/json",
                          "Authorization": f"Bearer {API_KEY}",
                          "HTTP-Referer": "https://bunka.ai/",
                          "X-Title": "Cultura Treccani validation"})
        _tls.s = s
    return _tls.s


def call_gemini(system, user, max_retries=4):
    body = {"model": MODEL,
            "messages": [{"role": "system", "content": system},
                         {"role": "user", "content": user}],
            "temperature": 0,
            "response_format": {"type": "json_object"}}
    last_err = ""
    for attempt in range(max_retries):
        try:
            r = _session().post(OPENROUTER_URL, json=body, timeout=180)
            if r.status_code in (408, 429, 500, 502, 503, 504):
                last_err = f"http_{r.status_code}"
                time.sleep(1.5 * 2 ** attempt)
                continue
            r.raise_for_status()
            return json.loads(r.json()["choices"][0]["message"]["content"])
        except (requests.RequestException, ValueError, KeyError) as e:
            last_err = f"{type(e).__name__}: {e}"
            time.sleep(1.5 * 2 ** attempt)
    raise RuntimeError(last_err)


STEP1_SYSTEM = """You are an expert historian reading a biography from the \
Dizionario Biografico degli Italiani (Treccani), written in Italian.

Your task: identify the ONE most granular location (a city, a region, or \
similar) associated with the individual's FLORUIT — the period during which \
they made their principal contribution — and the floruit period itself.

An individual may be linked to several locations during their active years: \
you must pick the single location best supported by the text as the place of \
their principal activity (not necessarily birthplace or deathplace).

Return STRICT JSON with exactly these keys:
  floruit_location        : most granular location name (English exonym if one exists, e.g. "Florence", "Rome")
  floruit_location_type   : "city" | "region" | "other"
  floruit_start           : integer year (negative = BCE)
  floruit_end             : integer year
  evidence_location_verbatim : verbatim passage from the text (original language) supporting the location
  evidence_location_en       : English translation of that passage
  evidence_floruit_verbatim  : verbatim passage supporting the floruit period (original language)
  evidence_floruit_en        : English translation of that passage
  reasoning               : 1-3 sentences in English

Rules:
- Verbatim passages MUST be copied exactly from the text (no paraphrase).
- If the same passage supports both location and floruit, it may be repeated.
- Reasoning and location name in English; verbatim quotes stay in the original language."""


def step1_user(row):
    text = pages[row["wikidata_id"]]["text"][:BIO_CHAR_CAP]
    return (f"Individual: {row['name_en']} ({row['wikidata_id']})\n\n"
            f"--- DBI BIOGRAPHY ---\n{text}\n--- END ---\n\n"
            "Extract the requested fields. JSON only.")


STEP1_PATH = CACHE_DIR / "step1_floruit_location.jsonl"
step1 = {}
if STEP1_PATH.exists():
    for line in STEP1_PATH.open():
        r = json.loads(line)
        step1[r["wikidata_id"]] = r["extraction"]

todo = [row for _, row in sample.iterrows() if row["wikidata_id"] not in step1]
if todo:
    with ThreadPoolExecutor(max_workers=5) as ex, STEP1_PATH.open("a") as f:
        futs = {ex.submit(call_gemini, STEP1_SYSTEM, step1_user(row)): row["wikidata_id"]
                for row in todo}
        for fut in tqdm(as_completed(futs), total=len(futs), desc="LLM step 1"):
            qid = futs[fut]
            step1[qid] = fut.result()
            f.write(json.dumps({"wikidata_id": qid, "extraction": step1[qid]},
                               ensure_ascii=False) + "\n")

assert len(step1) >= len(sample)
pd.DataFrame([{"wikidata_id": q, **step1[q]} for q in sample["wikidata_id"]])[
    ["wikidata_id", "floruit_location", "floruit_location_type",
     "floruit_start", "floruit_end"]]

LLM step 1:   0%|          | 0/10 [00:00<?, ?it/s]

,wikidata_id,floruit_location,floruit_location_type,floruit_start,floruit_end
0,Q956946,Rome,city,1294,1303
1,Q160424,Benevento,region,763,782
2,Q2436996,Venice,city,1432,1435
3,Q3620361,Florence,city,1395,1395
4,Q3766550,Vicenza,city,1595,1623
5,Q55226791,Siena,city,1494,1515
6,Q55226863,Munich,city,1671,1700
7,Q3770142,Naples,city,1764,1790
8,Q275985,Turin,city,1945,1952
9,Q1062517,Rome,city,1908,1913


## 5. LLM step 2 — map the location to a Cliopatria polity

For each individual the model chooses from the Cliopatria polities whose
period overlaps the extracted floruit.

In [6]:
def cliopatria_candidates(start, end):
    return con.execute("""
        SELECT polity_id, polity_name,
               min(from_year) AS from_year, max(to_year) AS to_year
        FROM polities_periods_cliopatria
        WHERE to_year >= ? AND from_year <= ?
        GROUP BY polity_id, polity_name
        ORDER BY polity_name
    """, [start, end]).df()


STEP2_SYSTEM = """You are an expert historical geographer.

Given a location, a floruit period, and the list of Cliopatria polities \
active during that period, choose the ONE polity that governed the location \
during the floruit period. If sovereignty changed during the period, choose \
the polity covering the largest share of the period.

Return STRICT JSON with exactly these keys:
  polity_id    : integer id copied from the candidate list
  polity_name  : name copied verbatim from the candidate list
  confidence   : "high" | "medium" | "low"
  reasoning    : 1-3 sentences in English

Rules:
- polity_id and polity_name MUST come from the candidate list, no invention.
- If no candidate plausibly governed the location, use polity_id = null and
  explain in reasoning."""


def step2_user(row, ext):
    cand = cliopatria_candidates(ext["floruit_start"], ext["floruit_end"])
    lines = "\n".join(f"id={r.polity_id} | {r.polity_name} | {r.from_year} to {r.to_year}"
                      for r in cand.itertuples())
    return (f"Location: {ext['floruit_location']}\n"
            f"Floruit period: {ext['floruit_start']} to {ext['floruit_end']}\n"
            f"Individual (context only): {row['name_en']}\n\n"
            f"--- CLIOPATRIA CANDIDATE POLITIES ({len(cand)}) ---\n{lines}\n--- END ---\n\n"
            "Choose the polity. JSON only.")


STEP2_PATH = CACHE_DIR / "step2_polity_mapping.jsonl"
step2 = {}
if STEP2_PATH.exists():
    for line in STEP2_PATH.open():
        r = json.loads(line)
        step2[r["wikidata_id"]] = r["mapping"]

todo = [row for _, row in sample.iterrows() if row["wikidata_id"] not in step2]
if todo:
    with ThreadPoolExecutor(max_workers=5) as ex, STEP2_PATH.open("a") as f:
        futs = {ex.submit(call_gemini, STEP2_SYSTEM,
                          step2_user(row, step1[row["wikidata_id"]])): row["wikidata_id"]
                for row in todo}
        for fut in tqdm(as_completed(futs), total=len(futs), desc="LLM step 2"):
            qid = futs[fut]
            step2[qid] = fut.result()
            f.write(json.dumps({"wikidata_id": qid, "mapping": step2[qid]},
                               ensure_ascii=False) + "\n")

assert len(step2) >= len(sample)
pd.DataFrame([{"wikidata_id": q, **step2[q]} for q in sample["wikidata_id"]])[
    ["wikidata_id", "polity_id", "polity_name", "confidence"]]

LLM step 2:   0%|          | 0/10 [00:00<?, ?it/s]

,wikidata_id,polity_id,polity_name,confidence
0,Q956946,419.0,Papal States,high
1,Q160424,402.0,Carolingian Empire,high
2,Q2436996,397.0,Republic of Venice,high
3,Q3620361,768.0,Republic of Florence,high
4,Q3766550,397.0,Republic of Venice,high
5,Q55226791,NaN,None,low
6,Q55226863,780.0,Duchy of Bavaria,high
7,Q3770142,1140.0,Kingdom of Naples (Napoleonic),high
8,Q275985,1428.0,Republic of Italy,high
9,Q1062517,350.0,Kingdom of Italy,high


## 6. Labeling TSV

One row per individual. `annot_*` columns are empty — to be filled by the
annotator (yes / no / notes). `auto_floruit_overlap_50pct` applies the rule:
the Cultura floruit is correct if it spans ≥ 50% of the extracted floruit.

In [7]:
def overlap_50pct(row):
    s, e = row["llm_floruit_start"], row["llm_floruit_end"]
    cs, ce = row["cultura_floruit_start"], row["cultura_floruit_end"]
    if pd.isna(s) or pd.isna(e) or e < s:
        return None
    ov = max(0, min(e, ce) - max(s, cs) + 1)
    return bool(ov >= 0.5 * (e - s + 1))


records = []
for _, row in sample.iterrows():
    qid = row["wikidata_id"]
    e1, e2 = step1[qid], step2[qid]
    records.append({
        "wikidata_id": qid,
        "name": row["name_en"],
        "dbi_url": row["dbi_url"],
        "region": row["region"],
        "period_bin": str(row["period_bin"]),
        "llm_floruit_location": e1["floruit_location"],
        "llm_floruit_location_type": e1["floruit_location_type"],
        "llm_floruit_start": e1["floruit_start"],
        "llm_floruit_end": e1["floruit_end"],
        "evidence_location_verbatim": e1["evidence_location_verbatim"],
        "evidence_location_en": e1["evidence_location_en"],
        "evidence_floruit_verbatim": e1["evidence_floruit_verbatim"],
        "evidence_floruit_en": e1["evidence_floruit_en"],
        "llm_step1_reasoning": e1["reasoning"],
        "llm_polity_id": e2["polity_id"],
        "llm_polity_name": e2["polity_name"],
        "llm_polity_confidence": e2["confidence"],
        "llm_step2_reasoning": e2["reasoning"],
        "cultura_polities": row["cultura_polities"],
        "cultura_floruit_start": row["floruit_period_start"],
        "cultura_floruit_end": row["floruit_period_end"],
        "annot_location_ok": "",
        "annot_floruit_ok": "",
        "annot_polity_ok": "",
        "annot_notes": "",
    })

out = pd.DataFrame(records)
out["auto_floruit_overlap_50pct"] = out.apply(overlap_50pct, axis=1)
assert out["wikidata_id"].is_unique and len(out) == 10

OUT_PATH = OUT_DIR / "treccani_validation_sample10.tsv"
out.to_csv(OUT_PATH, sep="\t", index=False)
print("saved:", OUT_PATH)
out[["name", "region", "period_bin", "llm_floruit_location",
     "llm_floruit_start", "llm_floruit_end", "llm_polity_name",
     "cultura_polities", "auto_floruit_overlap_50pct"]]

saved: /Users/charlesdedampierre/Desktop/Rsearch Folder/cultura/cultura_database/annotations/treccani_validation/treccani_validation_sample10.tsv


,name,region,period_bin,llm_floruit_location,llm_floruit_start,llm_floruit_end,llm_polity_name,cultura_polities,auto_floruit_overlap_50pct
0,John of Capua,South & Islands,<1300,Rome,1294,1303,Papal States,Papal States,True
1,Paul the Deacon,North,<1300,Benevento,763,782,Carolingian Empire,Alliance between Byzantine Empire and Khazaria...,True
2,Tito Livio Frulovisi,North,1300-1500,Venice,1432,1435,Republic of Venice,Papal States,True
3,Antonio da Pisa,Center,1300-1500,Florence,1395,1395,Republic of Florence,Republic of Pisa,True
4,Giovanni Battista Albanese,North,1500-1650,Vicenza,1595,1623,Republic of Venice,Republic of Venice,True
5,Litti Corbizzi,Center,1500-1650,Siena,1494,1515,None,Holy Roman Empire Minor States; Holy Roman Empire,True
6,Isabella Maria dal Pozzo,North,1650-1800,Munich,1671,1700,Duchy of Bavaria,Duchy of Bavaria,True
7,Giuseppe Bonecchi,Center,1650-1800,Naples,1764,1790,Kingdom of Naples (Napoleonic),Habsburg Monarchy,True
8,Natalia Ginzburg,South & Islands,1800+,Turin,1945,1952,Republic of Italy,Republic of Italy,True
9,Giulio Aristide Sartorio,Center,1800+,Rome,1908,1913,Kingdom of Italy,Kingdom of Italy,True
